**Classical model**

This notebook forms the layout for the first of the two models: the classical one. In both models, we're going to use the "California Housing Prices: Median house prices for California districts derived from the 1990 census" as it's well known and presents interesting challenges in cleaning the data to be suitable for both types of models. It's also used very frequently in learning machine learning. We start by importing important libraries and fetching the data set:

In [10]:
import numpy as np, json, time
from pathlib import Path
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import pandas as pd

Numpy is for numerical operations, sklearn for tuning, cleaning, and classifying the data (the real deal), and pandas is for dealing with datasets. Now we take a look at the data:

In [3]:
data = fetch_california_housing(as_frame= True)
df = data.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


To simplify everything, we choose just 4 features of the dataset to work with, which we chose as plausible predictors (median income, location and house age). The job of the classifier will be predicting whether the house price is above or below the median of all houses, where 1 indicates above and 0 indicates below. We will start by labeling the data:

In [4]:
FEATURES = ["MedInc", "Latitude", "Longitude", "HouseAge"]
y_all = (df["MedHouseVal"] > df["MedHouseVal"].median()).astype(int).values
X_all = df[FEATURES].values
print(X_all.shape, y_all.shape)
print("class balance:", np.bincount(y_all))

(20640, 4) (20640,)
class balance: [10323 10317]


The class balance is roughly 50/50, which is great. As the data is about 20k rows and the quantum model is too slow for that, we'll use 800 of the rows. We start by choosing these rows and stratifying them so that they're also roughly 50/50 on class balance:

In [5]:
SEED = 0
N_SAMPLES = 800
X_sub, _, y_sub, _ = train_test_split(X_all, y_all, train_size=N_SAMPLES, stratify=y_all, random_state=SEED)
print(X_sub.shape, np.bincount(y_sub))

(800, 4) [400 400]


Now we split our training and test split normally:

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.25, stratify=y_sub, random_state=SEED)
print("train:", X_train.shape, np.bincount(y_train))
print("test: ", X_test.shape, np.bincount(y_test))

train: (600, 4) [300 300]
test:  (200, 4) [100 100]


Now we standardize and transform the data into [0, pi] range to be able to use it in the quantum model. We use QuantileTransformer in order to give each entry a number between 0 and 1 first, then we multiply by required range [0, pi]:

In [7]:
qt = QuantileTransformer(n_quantiles=200, output_distribution="uniform")
qt.fit(X_train)                                

X_train_s = qt.transform(X_train) * np.pi
X_test_s = qt.transform(X_test) * np.pi

print("train range:", X_train_s.min(), X_train_s.max())
print("test range: ", X_test_s.min(), X_test_s.max())

SPLITS = Path("splits")
SPLITS.mkdir(exist_ok=True)
np.savez(SPLITS / f"seed{SEED}.npz",
         Xtr=X_train_s, Xte=X_test_s, ytr=y_train, yte=y_test)

train range: 0.0 3.141592653589793
test range:  0.0 3.141592653589793


Now we wrap the whole preparation (subsample, split, scale, save) into one function that takes a seed. A single split can be lucky or unlucky, so we'll repeat the process for 5 seeds and report the mean and standard deviation. The quantum notebook will load the exact same files, so both models are compared on identical data.

In [11]:
SPLITS = Path("splits")
SPLITS.mkdir(exist_ok=True)

def make_split(seed):
    X_sub, _, y_sub, _ = train_test_split(
        X_all, y_all, train_size=N_SAMPLES, stratify=y_all, random_state=seed)


    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sub, y_sub, test_size=0.25, stratify=y_sub, random_state=seed)

    qt = QuantileTransformer(n_quantiles=200, output_distribution="uniform")
    qt.fit(X_tr)
    X_tr_s = qt.transform(X_tr) * np.pi
    X_te_s = qt.transform(X_te) * np.pi

    #We here save the files to another folder to use them directly in the quantum model
    np.savez(SPLITS / f"seed{seed}.npz",
             Xtr=X_tr_s, Xte=X_te_s, ytr=y_tr, yte=y_te)

Now we generate the five splits and check each one: every file should have 600 training rows, 200 test rows, and balanced classes.

In [12]:
SEEDS = [0, 1, 2, 3, 4]
for s in SEEDS:
    make_split(s)
    d = np.load(SPLITS / f"seed{s}.npz")
    print(s, d["Xtr"].shape, d["Xte"].shape, np.bincount(d["ytr"]), np.bincount(d["yte"]),
          "range:", round(d["Xtr"].min(), 2), round(d["Xtr"].max(), 2))

0 (600, 4) (200, 4) [300 300] [100 100] range: 0.0 3.14
1 (600, 4) (200, 4) [300 300] [100 100] range: 0.0 3.14
2 (600, 4) (200, 4) [300 300] [100 100] range: 0.0 3.14
3 (600, 4) (200, 4) [300 300] [100 100] range: 0.0 3.14
4 (600, 4) (200, 4) [300 300] [100 100] range: 0.0 3.14


Now the classical models. We use four, each with a role:

- **Logistic regression**: a simple linear baseline.
- **SVM (RBF kernel)**: a strong classical reference, tuned with 5-fold cross-validation on the training data only.
- **Small MLP (6 hidden neurons)**: about 37 parameters, deliberately close to the quantum circuit's parameter count for a fair comparison.
- **Large MLP (32, 32)**: shows what extra capacity buys.

`count_params` counts the trainable parameters of each model, so we can report model size next to accuracy.

In [ ]:
def get_models(seed):
    #unfitted models for each seed
    return {
        "logreg": LogisticRegression(max_iter=1000),
        "svm_rbf": GridSearchCV(SVC(), {"C": [0.1, 1, 10], "gamma": ["scale", 0.1, 1]}, cv=5),
        "mlp_small": MLPClassifier((6,), max_iter=3000, random_state=seed),
        "mlp_large": MLPClassifier((32, 32), max_iter=3000, random_state=seed),
    }

def count_params(model):
    est = getattr(model, "best_estimator_", model)       # unwrap GridSearchCV
    if hasattr(est, "coefs_"):                           # MLP
        return sum(w.size for w in est.coefs_) + sum(b.size for b in est.intercepts_)
    if hasattr(est, "coef_"):                            # logistic regression
        return est.coef_.size + est.intercept_.size
    return None                                          # SVM: not comparable

Now we train every model on every seed. Each model is loaded from the saved `.npz` files (the same code path the quantum notebook will use), and for each run we record train accuracy, test accuracy, fit time, and parameter count.

In [14]:
results = {}

for seed in SEEDS:
    d = np.load(SPLITS / f"seed{seed}.npz")
    Xtr, Xte, ytr, yte = d["Xtr"], d["Xte"], d["ytr"], d["yte"]

    for name, model in get_models(seed).items():
        t0 = time.time()
        model.fit(Xtr, ytr)
        fit_time = time.time() - t0

        results.setdefault(name, []).append({
            "seed": seed,
            "train_acc": float(model.score(Xtr, ytr)),
            "test_acc": float(model.score(Xte, yte)),
            "fit_time_s": float(fit_time),
            "n_params": count_params(model),
        })
    print("finished seed", seed)

finished seed 0
finished seed 1
finished seed 2
finished seed 3
finished seed 4


We save the results to a JSON file. The quantum notebook will write `results/quantum.json` in exactly the same format, so a single cell at the end can compare both models.

In [15]:
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

with open(RESULTS / "classical.json", "w") as f:
    json.dump(results, f, indent=2)

Finally, we summarize the results: mean and standard deviation of test accuracy across the 5 seeds, plus train accuracy (to spot overfitting), fit time, and parameter count.

In [16]:
rows = [{"model": name, **r} for name, runs in results.items() for r in runs]
res = pd.DataFrame(rows)

summary = res.groupby("model").agg(
    test_mean=("test_acc", "mean"),
    test_std=("test_acc", "std"),
    train_mean=("train_acc", "mean"),
    time_mean_s=("fit_time_s", "mean"),
    n_params=("n_params", "first"),
).round(3)

summary.sort_values("test_mean", ascending=False)

,test_mean,test_std,train_mean,time_mean_s,n_params
model,,,,,
svm_rbf,0.818,0.020,0.879,0.631,NaN
mlp_large,0.811,0.041,0.860,1.827,1249.0
logreg,0.800,0.015,0.814,0.010,5.0
mlp_small,0.795,0.029,0.799,0.704,37.0
